# Notebook 1 — Web Scraping Berita Multi-Portal Indonesia

**Sumber:** Detik.com · Kompas.com · Tribunnews.com · CNN Indonesia · Antara News · Liputan6.com  
**Tujuan:** Dataset berita berbahasa Indonesia untuk Topic Modelling  
**Output:** `dataset_berita.csv` (target ≥ 10.000 baris)

**Etika scraping:**
- Delay acak 1–3 detik antar request agar tidak membebani server
- User-Agent diidentifikasi sebagai browser umum
- Data hanya untuk keperluan akademik / riset

## 1.1 Install Dependensi


In [1]:
!pip install requests beautifulsoup4 pandas lxml -q

## 1.2 Import Library


In [2]:
import csv
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

## 1.3 Konfigurasi Global


In [3]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
}

MAX_HALAMAN = 70       # halaman indeks per kategori per portal
MIN_PANJANG = 150      # buang artikel < N karakter
DELAY_MIN   = 1.0
DELAY_MAX   = 3.0
OUTPUT_FILE = "dataset_berita.csv"

## 1.4 Konfigurasi Per Portal

Setiap portal punya `indeks_url` template (gunakan `{base}` dan `{page}`) serta key `artikel_parser` yang menunjuk ke fungsi parser yang sesuai.


In [4]:
PORTAL_CONFIG = {

    # DETIK — pagination: /indeks?page=N
    "detik": {
        "indeks_url":     "{base}?page={page}",
        "artikel_parser": "detik",
        "kategori": {
            "ekonomi":   "https://finance.detik.com/indeks",
            "teknologi": "https://inet.detik.com/indeks",
            "olahraga":  "https://sport.detik.com/indeks",
            "politik":   "https://news.detik.com/indeks",
            "hiburan":   "https://hot.detik.com/indeks",
            "otomotif":  "https://oto.detik.com/indeks",
        },
    },

    # # KOMPAS — indeks.kompas.com/?site=money&page=N
    "kompas": {
        "indeks_url":     "{base}&page={page}",
        "artikel_parser": "kompas",
        "kategori": {
            "ekonomi":   "https://indeks.kompas.com/?site=money",
            "teknologi": "https://indeks.kompas.com/?site=tekno",
            "olahraga":  "https://indeks.kompas.com/?site=bola",
            "politik":   "https://indeks.kompas.com/?site=nasional",
            "hiburan":   "https://indeks.kompas.com/?site=entertainment",
            "sains":     "https://indeks.kompas.com/?site=sains",
        },
    },

    # # TRIBUN — index-news/{kategori}?page=N
    "tribun": {
        "indeks_url":     "{base}?page={page}",
        "artikel_parser": "tribun",
        "kategori": {
            "ekonomi":   "https://www.tribunnews.com/index-news/bisnis",
            "teknologi": "https://www.tribunnews.com/index-news/techno",
            "olahraga":  "https://www.tribunnews.com/index-news/sport",
            "politik":   "https://www.tribunnews.com/index-news/nasional",
            "hiburan":   "https://www.tribunnews.com/index-news/seleb",
            "lifestyle":  "https://www.tribunnews.com/index-news/lifestyle",
        },
    },

    # # CNN INDONESIA — category root + ?page=N
    "cnnindonesia": {
        "indeks_url":     "{base}?page={page}",
        "artikel_parser": "cnn",
        "kategori": {
            "ekonomi":   "https://www.cnnindonesia.com/ekonomi",
            "teknologi": "https://www.cnnindonesia.com/teknologi",
            "olahraga":  "https://www.cnnindonesia.com/olahraga",
            "politik":   "https://www.cnnindonesia.com/nasional",
            "hiburan":   "https://www.cnnindonesia.com/hiburan",
        },
    },

    # ANTARA NEWS — path pagination: /kategori/2, /kategori/3 (Tempo diganti karena paywall)
    "antara": {
        "indeks_url":     "{base}/{page}",
        "artikel_parser": "antara",
        "kategori": {
            "ekonomi":   "https://www.antaranews.com/ekonomi",
            "teknologi": "https://www.antaranews.com/tekno",
            "olahraga":  "https://www.antaranews.com/olahraga",
            "politik":   "https://www.antaranews.com/politik",
            "hiburan":   "https://www.antaranews.com/hiburan",
            "sains":     "https://www.antaranews.com/humaniora",
        },
    },

    # LIPUTAN6 — pagination: /{kategori}/indeks?page=N
    "liputan6": {
        "indeks_url":     "{base}?page={page}",
        "artikel_parser": "liputan6",
        "kategori": {
            "nasional":  "https://www.liputan6.com/news/indeks",
            "ekonomi":   "https://www.liputan6.com/bisnis/indeks",
            "teknologi": "https://www.liputan6.com/tekno/indeks",
            "olahraga":  "https://www.liputan6.com/bola/indeks",
            "hiburan":   "https://www.liputan6.com/showbiz/indeks",
            "otomotif":  "https://www.liputan6.com/otomotif/indeks",
            "lifestyle": "https://www.liputan6.com/lifestyle/indeks",
        },
    },
}

## 1.5 Utilitas HTTP


In [5]:
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def get_page(url: str, retries: int = 3):
    """GET halaman dan kembalikan BeautifulSoup, atau None jika gagal."""
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=12)
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [!] {url} — {e} (retry {attempt+1}/{retries}, tunggu {wait}s)")
            time.sleep(wait)
    return None


def tidy_text(text: str) -> str:
    """Bersihkan whitespace berlebih."""
    return re.sub(r"\s+", " ", text).strip()

## 1.6 Parser Indeks

Fungsi-fungsi di bawah mengekstrak **daftar URL artikel** dari halaman indeks masing-masing portal.


In [6]:
def _collect_links(soup, css_selectors, domain_filter=None, path_filter=None):
    """Helper: try CSS selectors first, fall back to all <a> tags filtered by domain/path."""
    links = []
    for sel in css_selectors:
        for a in soup.select(sel):
            href = a.get("href", "")
            if href.startswith("http"):
                links.append(href)
    if not links:
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if not href.startswith("http"):
                continue
            if domain_filter and domain_filter not in href:
                continue
            if path_filter and path_filter not in href:
                continue
            links.append(href)
    return links


def ambil_link_detik(soup):
    return _collect_links(
        soup,
        ["article a[href]"],
        domain_filter="detik.com",
        path_filter="/read/",
    )


def ambil_link_kompas(soup):
    return _collect_links(
        soup,
        ["div.latest__wrap a.artikel__link", "h3.article__title a", "article a[href]"],
        domain_filter="kompas.com",
        path_filter="/read/",
    )


def ambil_link_tribun(soup):
    return _collect_links(
        soup,
        ["div.lsi h3 a", "div.side-articles h3 a", "li.clearfix h3 a", "h3 a[href]"],
        domain_filter="tribunnews.com",
        path_filter=None,
    )


def ambil_link_cnn(soup):
    return _collect_links(
        soup,
        ["article.container__item a.container__link", "h3.title a", ".list-item a"],
        domain_filter="cnnindonesia.com",
        path_filter=None,
    )


def ambil_link_antara(soup):
    return _collect_links(
        soup,
        ["div.simple-list a[href]", "h2 a[href]", "h3 a[href]"],
        domain_filter="antaranews.com",
        path_filter="/berita/",
    )


def ambil_link_liputan6(soup):
    return _collect_links(
        soup,
        ["a[href*='/read/']", "h4.articles__title a", "article a[href]"],
        domain_filter="liputan6.com",
        path_filter="/read/",
    )


INDEKS_PARSERS = {
    "detik":    ambil_link_detik,
    "kompas":   ambil_link_kompas,
    "tribun":   ambil_link_tribun,
    "cnn":      ambil_link_cnn,
    "antara":   ambil_link_antara,
    "liputan6": ambil_link_liputan6,
}

## 1.7 Parser Artikel

Fungsi-fungsi di bawah mengekstrak **judul, tanggal, dan isi** dari halaman detail artikel.


In [7]:
def parse_detik(soup, url, kategori):
    judul_tag = soup.find("h1", class_="detail__title") or soup.find("h1")
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = soup.find("div", class_="detail__date")
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = soup.find("div", class_="detail__body-text") or soup.find("div", class_="itp_bodycontent")
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "detik", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


def parse_kompas(soup, url, kategori):
    judul_tag = soup.find("h1", class_="read__title") or soup.find("h1")
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = soup.find("div", class_="read__time")
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = soup.find("div", class_="read__content") or soup.find("div", class_="clearfix")
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "kompas", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


def parse_tribun(soup, url, kategori):
    judul_tag = soup.find("h1", id="arttitle") or soup.find("h1")
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = soup.find("time") or soup.find("span", class_="date")
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = (soup.find("div", class_="side-article txt-article") or
            soup.find("div", id="article-body") or
            soup.find("div", class_="txt-article"))
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "tribun", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


def parse_cnn(soup, url, kategori):
    judul_tag = soup.find("h1", class_="title") or soup.find("h1")
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = soup.find("div", class_="date") or soup.find("time")
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = soup.find("div", class_="detail-text") or soup.find("section", class_="detail-wrap")
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "cnnindonesia", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


def parse_antara(soup, url, kategori):
    judul_tag = soup.find("h1") or soup.find("h2")
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = (soup.find("span", class_="article-date") or
               soup.find("time") or
               soup.find("li", class_="text-muted"))
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = (soup.find("div", class_="wrap-bodynews") or
            soup.find("div", attrs={"id": "article-body"}) or
            soup.find("div", class_="post-content") or
            soup.find("article"))
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "antara", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


def parse_liputan6(soup, url, kategori):
    judul_tag = (soup.find("h1", class_=lambda c: c and "title" in c) or
                 soup.find("h1"))
    judul = tidy_text(judul_tag.get_text()) if judul_tag else ""
    tgl_tag = soup.find("time") or soup.find("span", class_=lambda c: c and "date" in c)
    tanggal = tidy_text(tgl_tag.get_text()) if tgl_tag else ""
    body = (soup.find("div", class_=lambda c: c and "article-content" in c) or
            soup.find("div", attrs={"data-component": "article-body"}) or
            soup.find("div", class_="read-page--content") or
            soup.find("article"))
    if not body: return None
    isi = " ".join(tidy_text(p.get_text()) for p in body.find_all("p"))
    return {"portal": "liputan6", "kategori": kategori, "judul": judul,
            "tanggal": tanggal, "isi": isi, "url": url}


ARTIKEL_PARSERS = {
    "detik":    parse_detik,
    "kompas":   parse_kompas,
    "tribun":   parse_tribun,
    "cnn":      parse_cnn,
    "antara":   parse_antara,
    "liputan6": parse_liputan6,
}

## 1.8 Fungsi Inti: Crawl Satu Portal


In [8]:
CSV_FIELDNAMES = ["portal", "kategori", "judul", "tanggal", "isi", "url"]


def crawl_portal(nama_portal, config, csv_writer, max_halaman=MAX_HALAMAN, url_awal=None):
    """Crawl seluruh kategori pada satu portal; tulis langsung ke CSV via csv_writer."""
    parser_key   = config["artikel_parser"]
    indeks_fn    = INDEKS_PARSERS[parser_key]
    artikel_fn   = ARTIKEL_PARSERS[parser_key]
    url_template = config["indeks_url"]

    jumlah = 0
    url_sudah_dikunjungi = set(url_awal) if url_awal else set()

    for kategori, base_url in config["kategori"].items():
        print(f"\n  [{nama_portal.upper()}] Kategori: {kategori}")

        for hal in range(1, max_halaman + 1):
            url_indeks = base_url if hal == 1 else url_template.format(base=base_url, page=hal)

            soup = get_page(url_indeks)
            if soup is None:
                break

            semua_links = indeks_fn(soup)
            if not semua_links:          # halaman benar-benar kosong / parser gagal
                break

            baru = [l for l in set(semua_links) if l not in url_sudah_dikunjungi]
            print(f"    Hal {hal}: {len(baru)} baru / {len(semua_links)} total")

            for url_art in baru:
                url_sudah_dikunjungi.add(url_art)
                soup_art = get_page(url_art)
                if soup_art is None:
                    continue

                data = artikel_fn(soup_art, url_art, kategori)
                if data and len(data.get("isi", "")) >= MIN_PANJANG:
                    csv_writer.writerow(data)
                    jumlah += 1
                    print(f"      ✓ [{jumlah:>5}] {data['judul'][:65]}...")

                time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    return jumlah

## 1.9  Eksekusi Scraping

> ⚠️ **Estimasi waktu:** ± 7–10 jam untuk semua portal dengan `MAX_HALAMAN = 20` (~11.000 artikel).  
> Kurangi `MAX_HALAMAN` di sel 1.3 jika ingin uji cepat (mis. `MAX_HALAMAN = 1`).  
> Data langsung tersimpan ke CSV per artikel — aman dihentikan kapan saja dan dilanjutkan manual.

In [ ]:
import csv as _csv

# Baca URL yang sudah di-scrape agar tidak duplikat
try:
    df_existing = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig", usecols=["url"])
    url_lama = set(df_existing["url"].dropna())
    total = len(url_lama)
    print(f"[+] Resume: {total} artikel sudah ada, melanjutkan...")
except FileNotFoundError:
    url_lama = set()
    total = 0
    print("[+] Mulai scraping baru...")

print(f"[+] Mulai    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[+] Portal   : {', '.join(PORTAL_CONFIG.keys())}")
print(f"[+] Max hal  : {MAX_HALAMAN} per kategori per portal")
print(f"[+] Output   : {OUTPUT_FILE}\n")

# Buka CSV dalam mode append; tulis header hanya jika file baru
file_baru = total == 0
with open(OUTPUT_FILE, "a", newline="", encoding="utf-8-sig") as f:
    writer = _csv.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
    if file_baru:
        writer.writeheader()

    for nama_portal, config in PORTAL_CONFIG.items():
        print(f"\n{'='*55}")
        print(f"  PORTAL: {nama_portal.upper()}")
        print(f"{'='*55}")
        n = crawl_portal(nama_portal, config, writer, MAX_HALAMAN, url_awal=url_lama)
        total += n
        f.flush()
        print(f"  Subtotal {nama_portal}: {n} artikel baru  (total: {total})")

print(f"\n[+] Selesai : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[+] Total artikel tersimpan: {total}")
print(f"[+] File    : {OUTPUT_FILE}")

[+] Resume: 9575 artikel sudah ada, melanjutkan...
[+] Mulai    : 2026-05-28 11:32:23
[+] Portal   : detik, kompas, tribun, cnnindonesia, antara, liputan6
[+] Max hal  : 70 per kategori per portal
[+] Output   : dataset_berita.csv


  PORTAL: DETIK

  [DETIK] Kategori: ekonomi
    Hal 1: 4 baru / 40 total
    Hal 2: 3 baru / 40 total
    Hal 3: 0 baru / 40 total
    Hal 4: 0 baru / 40 total
    Hal 5: 2 baru / 40 total
    Hal 6: 3 baru / 40 total
    Hal 7: 1 baru / 40 total
    Hal 8: 0 baru / 40 total
    Hal 9: 0 baru / 40 total
    Hal 10: 1 baru / 40 total
    Hal 11: 1 baru / 40 total
    Hal 12: 3 baru / 40 total
    Hal 13: 3 baru / 40 total
    Hal 14: 2 baru / 40 total
    Hal 15: 2 baru / 40 total
    Hal 16: 4 baru / 40 total
    Hal 17: 4 baru / 40 total
    Hal 18: 1 baru / 40 total
    Hal 19: 2 baru / 40 total
    Hal 20: 3 baru / 40 total
    Hal 21: 2 baru / 40 total
    Hal 22: 1 baru / 40 total
    Hal 23: 1 baru / 40 total
    Hal 24: 1 baru / 40 total
    Hal 25:

## 1.10 Simpan ke CSV


In [3]:
df = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")

before = len(df)
df.drop_duplicates(subset="url", inplace=True)
df.reset_index(drop=True, inplace=True)
after = len(df)

if before != after:
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"[+] Duplikat dihapus : {before - after} baris — file diperbarui")
else:
    print(f"[+] Tidak ada duplikat")

print(f"[+] Dataset          : {OUTPUT_FILE}")
print(f"[+] Shape final      : {df.shape}")
df.head()

NameError: name 'OUTPUT_FILE' is not defined

## 1.11 Statistik Dataset


In [ ]:
print("Distribusi per Portal:")
print(df["portal"].value_counts().to_string())

print("\nDistribusi per Kategori:")
print(df["kategori"].value_counts().to_string())

print(f"\nRata-rata panjang artikel: {df['isi'].str.len().mean():.0f} karakter")

Distribusi per Portal:


NameError: name 'df' is not defined

---

**Lanjut ke `02_preprocessing.ipynb`** untuk pembersihan teks.
